# Truthprint W5 — wide-coverage neural invariant parser

**Goal (review weakness W5):** replace the closed-domain *lexicon* extractor with a
real **neural** parser (an instruction LLM prompted to emit the nine invariant
fields as JSON), and measure it head-to-head against the lexicon on the same real
NLLB translations **and** on open-domain sentences the lexicon cannot handle.

**No manual annotation needed** (open-domain gold is bundled below). Run all cells;
the last writes `w5_results.zip` — send it back.

Recommended: Kaggle/Colab **GPU T4**, Internet **On**.


## Cell 1 — environment + clone + install


In [ ]:
import os, sys, subprocess
BASE = '/kaggle/working' if os.path.isdir('/kaggle/working') else ('/content' if os.path.isdir('/content') else os.getcwd())
REPO = os.path.join(BASE,'truthprint')
if not os.path.isdir(REPO):
    subprocess.run(['git','clone','--depth','1','https://github.com/leemgs/truthprint',REPO], check=True)
sys.path.insert(0, os.path.join(REPO,'code'))
WORK = os.path.join(BASE,'w5_work'); os.makedirs(WORK, exist_ok=True)
subprocess.run([sys.executable,'-m','pip','install','-q','transformers','sentencepiece','accelerate','torch','--upgrade'], check=False)
import torch; print('cuda:', torch.cuda.is_available())


## Cell 2 — evaluation sentences: closed template + open domain
`template` reuses the closed-domain facts; `open` are hand-labeled open-domain
facts (news/science/policy) with out-of-vocabulary wording the lexicon misses.


In [ ]:
import json, random
from truthprint import challenge as ch
rng = random.Random(41)
# --- closed template facts (gold = ext_invariants) ---
tmpl = []
for i in range(40):
    f = ch.sample_fact(rng)
    tmpl.append({'sent_id':f'T{i}','domain':'template','en':ch.realize(f, 0, 1),
                 'gold': ch.ext_invariants(f)})
# --- open-domain facts with hand gold over the 9 fields ---
OPEN = [
 {'en':'The central bank did not raise interest rates last quarter.',
  'gold':{'agent':'the central bank','patient':'interest rates','predicate':'RAISE','polarity':'negative','quantity':1,'time_dir':'previous','modality':'asserted','attribution':'none','causation':'none'}},
 {'en':'Researchers reported that the vaccine reduced infections in order to curb the outbreak.',
  'gold':{'agent':'researchers','patient':'infections','predicate':'REDUCE','polarity':'positive','quantity':1,'time_dir':'previous','modality':'asserted','attribution':'report','causation':'purpose'}},
 {'en':'According to the ministry, three factories must close because of the flood.',
  'gold':{'agent':'the ministry','patient':'factories','predicate':'CLOSE','polarity':'positive','quantity':3,'time_dir':'previous','modality':'necessary','attribution':'report','causation':'cause'}},
 {'en':'The satellite may have detected two storms forming over the ocean.',
  'gold':{'agent':'the satellite','patient':'storms','predicate':'DETECT','polarity':'positive','quantity':2,'time_dir':'previous','modality':'possible','attribution':'none','causation':'none'}},
 {'en':'The court will overturn the ruling next week to protect the tenants.',
  'gold':{'agent':'the court','patient':'the ruling','predicate':'OVERTURN','polarity':'positive','quantity':1,'time_dir':'following','modality':'asserted','attribution':'none','causation':'purpose'}},
 {'en':'The vendor confirmed that five servers were not patched.',
  'gold':{'agent':'the vendor','patient':'servers','predicate':'PATCH','polarity':'negative','quantity':5,'time_dir':'previous','modality':'asserted','attribution':'vendor','causation':'none'}},
]
for i,o in enumerate(OPEN): o['sent_id']=f'O{i}'; o['domain']='open'
items = tmpl + OPEN
print('eval sentences:', len(items), '(template', len(tmpl), '+ open', len(OPEN), ')')


## Cell 3 — real NLLB translation into 6 conditions


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
NLLB='facebook/nllb-200-distilled-600M'; device='cuda' if torch.cuda.is_available() else 'cpu'
ntok=AutoTokenizer.from_pretrained(NLLB); nmodel=AutoModelForSeq2SeqLM.from_pretrained(NLLB).to(device).eval()
L={'ko':'kor_Hang','hi':'hin_Deva','zh':'zho_Hans','ar':'arb_Arab','de':'deu_Latn','en':'eng_Latn'}
def tr(text,src,tgt):
    ntok.src_lang=L[src]; enc=ntok(text,return_tensors='pt',truncation=True,max_length=200).to(device)
    bos=ntok.convert_tokens_to_ids(L[tgt])
    with torch.no_grad(): out=nmodel.generate(**enc,forced_bos_token_id=bos,max_length=220)
    return ntok.batch_decode(out,skip_special_tokens=True)[0]
rows=[]   # one per (sentence, condition)
for it in items:
    en=it['en']
    rows.append({**it,'condition':'clean','lang':'en','text':en})
    rows.append({**it,'condition':'rt','lang':'en','text':tr(tr(en,'en','ko'),'ko','en')})
    for c in ['ko','hi','zh','ar','de']:
        rows.append({**it,'condition':c,'lang':c,'text':tr(en,'en',c)})
print('translated rows:', len(rows))


## Cell 4 — neural extractor backend (instruction LLM) + run
Swap `LLM_ID` for any instruct model. The backend is just `fn(prompt)->str`.


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
LLM_ID='Qwen/Qwen2.5-1.5B-Instruct'   # any instruct model works; smaller = faster
ltok=AutoTokenizer.from_pretrained(LLM_ID)
lmodel=AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype='auto', device_map='auto')
gen=pipeline('text-generation', model=lmodel, tokenizer=ltok, max_new_tokens=160, do_sample=False)
def backend(prompt):
    msgs=[{'role':'system','content':'You extract structured meaning as compact JSON.'},{'role':'user','content':prompt}]
    text=ltok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    out=gen(text, return_full_text=False)[0]['generated_text']
    return out
from truthprint.neural_parser import NeuralInvariantExtractor
ext=NeuralInvariantExtractor(backend)
out_path=os.path.join(WORK,'neural_parser_outputs.jsonl')
with open(out_path,'w',encoding='utf-8') as f:
    for i,r in enumerate(rows):
        pred=ext.extract(r['text'], r['lang'])
        f.write(json.dumps({'sent_id':r['sent_id'],'lang':r['lang'],'condition':r['condition'],
                            'domain':r['domain'],'text':r['text'],'gold':r['gold'],'pred':pred}, ensure_ascii=False)+'\n')
        if i%50==0: print('...', i, '/', len(rows))
print('wrote', out_path)


## Cell 5 — score (neural vs lexicon head-to-head) + zip


In [ ]:
import subprocess as sp, shutil
sp.run([sys.executable, os.path.join(REPO,'code','scripts','eval_neural_parser.py'), out_path, '--out', WORK], check=False)
zp=shutil.make_archive(os.path.join(BASE,'w5_results'),'zip',WORK); print('created:', zp)
md=os.path.join(WORK,'neural_parser.md')
print(open(md,encoding='utf-8').read() if os.path.exists(md) else 'scorer output missing')
